# SAST Baseline Evaluation (Semgrep + SonarQube)

Evaluates industry-standard Static Application Security Testing tools against the
SVD-Benchmark, reproducing the localization comparison.

| Tool | Metric reported | Paper result |
|------|-----------------|-------------|
| Semgrep | Top-1 Accuracy | 79% |
| SonarQube | Top-1 Accuracy | 66% |

**Evaluation protocol:**
- Each code snippet is written to a temporary `.java` file.
- The SAST tool scans it and reports flagged line numbers.
- A HIT is recorded when the flagged line number matches the ground-truth vulnerable line.
- Only vulnerable samples are evaluated for Top-1 Accuracy (benign samples assess
  false positive rate separately).

**Requirements:**
- Semgrep: `pip install semgrep`
- SonarQube: requires a running SonarQube server (see setup instructions below).


In [ ]:
# !pip install semgrep scikit-learn tqdm --quiet
import subprocess, tempfile, os, re, json
import pandas as pd
from tqdm import tqdm
import sys
sys.path.insert(0, ".")
from metrics_utils import compute_classification_metrics, parse_model_output


## 1. Load SVD-Benchmark

In [ ]:
benchmark_df = pd.read_csv("Evaluation-Benchmark/SVD-Benchmark.csv")
print(f"Loaded {len(benchmark_df)} samples")
print(benchmark_df["Status"].value_counts().to_string())


## 2. Semgrep Evaluation

**Tool:** Semgrep (~13.7k GitHub stars) — pattern-matching SAST for Java.
**Config:** `--config=auto` uses Semgrep's curated rule registry (Java security rules).

In [ ]:
def run_semgrep(code_snippet: str, ground_truth_line_no: int) -> dict:
    """
    Write code to a temp .java file, run Semgrep, and check if the flagged
    line number matches the ground-truth vulnerable line number.

    Returns a dict with:
        hit       : bool — did Semgrep flag the correct line?
        flagged   : list[int] — all line numbers Semgrep flagged
        raw_output: str — Semgrep JSON output
    """
    # Command: semgrep --config=auto --json <file>
    with tempfile.NamedTemporaryFile(suffix=".java", mode="w",
                                     delete=False, prefix="vulnlocai_") as f:
        f.write(code_snippet)
        tmp_path = f.name

    try:
        result = subprocess.run(
            ["semgrep", "--config=auto", "--json", tmp_path],
            capture_output=True, text=True, timeout=60,
        )
        raw = result.stdout
        try:
            data = json.loads(raw)
            flagged_lines = [r["start"]["line"] for r in data.get("results", [])]
        except json.JSONDecodeError:
            flagged_lines = []

        hit = int(ground_truth_line_no) in flagged_lines
        return {"hit": hit, "flagged": flagged_lines, "raw_output": raw}
    except subprocess.TimeoutExpired:
        return {"hit": False, "flagged": [], "raw_output": "TIMEOUT"}
    except FileNotFoundError:
        return {"hit": False, "flagged": [], "raw_output": "SEMGREP_NOT_FOUND"}
    finally:
        os.unlink(tmp_path)


# ── Run on vulnerable samples only (Top-1 Accuracy) ──────────────────────────
vuln_df = benchmark_df[benchmark_df["Status"] == "vulnerable"].copy()
benign_df = benchmark_df[benchmark_df["Status"] == "benign"].copy()

semgrep_results = []
for _, row in tqdm(vuln_df.iterrows(), total=len(vuln_df), desc="Semgrep (vulnerable)"):
    res = run_semgrep(str(row["Code Snippet"]), int(row["Line Number"]))
    semgrep_results.append(res)

vuln_df["semgrep_hit"]     = [r["hit"]     for r in semgrep_results]
vuln_df["semgrep_flagged"] = [r["flagged"] for r in semgrep_results]

# Top-1 Accuracy on vulnerable samples
top1_vuln = vuln_df["semgrep_hit"].sum() / len(vuln_df)

# False positive rate on benign samples
benign_results = []
for _, row in tqdm(benign_df.iterrows(), total=len(benign_df), desc="Semgrep (benign)"):
    res = run_semgrep(str(row["Code Snippet"]), -1)
    benign_results.append(len(res["flagged"]) > 0)
fp_rate = sum(benign_results) / len(benign_df)

print(f"\nSemgrep Results:")
print(f"  Top-1 Accuracy (vulnerable): {top1_vuln*100:.1f}%")
print(f"  False Positive Rate (benign): {fp_rate*100:.1f}%")
print(f"  (Paper reports 79% Top-1 Accuracy)")


## 3. SonarQube Evaluation

**Tool:** SonarQube (~10.1k GitHub stars) — enterprise-grade continuous inspection.

**Setup required:**
```bash
# 1. Download and start SonarQube Community Edition
docker run -d --name sonarqube -p 9000:9000 sonarqube:community

# 2. Install sonar-scanner
brew install sonar-scanner       # macOS
apt-get install sonar-scanner    # Ubuntu

# 3. Create a SonarQube project and get a token at http://localhost:9000
```

Set `SONAR_TOKEN` and `SONAR_HOST` below before running.


In [ ]:
SONAR_HOST  = os.getenv("SONAR_HOST", "http://localhost:9000")
SONAR_TOKEN = os.getenv("SONAR_TOKEN", "")
SONAR_PROJECT_KEY = "vulnlocai-eval"

def run_sonarqube(code_snippet: str, project_key: str,
                  ground_truth_line_no: int) -> dict:
    """
    Write code to a temp project directory, run sonar-scanner, fetch issues via
    the SonarQube API, and check if any flagged line matches the ground truth.

    Requires a running SonarQube server (see setup instructions above).
    """
    if not SONAR_TOKEN:
        return {"hit": False, "flagged": [], "raw_output": "SONAR_TOKEN_NOT_SET"}

    import requests, shutil

    # Create temp project directory
    tmp_dir = tempfile.mkdtemp(prefix="vulnlocai_sonar_")
    src_dir = os.path.join(tmp_dir, "src")
    os.makedirs(src_dir, exist_ok=True)

    java_file = os.path.join(src_dir, "VulnSnippet.java")
    with open(java_file, "w") as f:
        # Wrap snippet in a minimal compilable class if needed
        if "class " not in code_snippet:
            f.write(f"public class VulnSnippet {{\n{code_snippet}\n}}")
        else:
            f.write(code_snippet)

    props_file = os.path.join(tmp_dir, "sonar-project.properties")
    with open(props_file, "w") as f:
        f.write(f"sonar.projectKey={project_key}\n")
        f.write(f"sonar.sources=src\n")
        f.write(f"sonar.java.source=11\n")

    try:
        subprocess.run(
            ["sonar-scanner",
             f"-Dsonar.host.url={SONAR_HOST}",
             f"-Dsonar.login={SONAR_TOKEN}"],
            cwd=tmp_dir, capture_output=True, timeout=120,
        )
        # Fetch issues from API
        resp = requests.get(
            f"{SONAR_HOST}/api/issues/search",
            params={"componentKeys": project_key, "ps": 500},
            auth=(SONAR_TOKEN, ""),
        )
        data = resp.json()
        flagged_lines = [
            int(issue["textRange"]["startLine"])
            for issue in data.get("issues", [])
            if "textRange" in issue
        ]
        hit = int(ground_truth_line_no) in flagged_lines
        return {"hit": hit, "flagged": flagged_lines, "raw_output": str(data)}
    except Exception as e:
        return {"hit": False, "flagged": [], "raw_output": str(e)}
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


print("SonarQube evaluation ready.")
print(f"  Host:  {SONAR_HOST}")
print(f"  Token: {'SET' if SONAR_TOKEN else 'NOT SET — set SONAR_TOKEN env variable'}")


## 4. Save Results

In [ ]:
vuln_df.to_csv("Results/Semgrep_predictions.csv", index=False)

sast_summary = {
    "Semgrep": {
        "Top1_Accuracy": round(top1_vuln * 100, 1),
        "FP_Rate":       round(fp_rate * 100, 1),
    }
}
with open("Results/SAST_baselines_metrics.json", "w") as f:
    json.dump(sast_summary, f, indent=2)
print("Saved to Results/")
print(json.dumps(sast_summary, indent=2))
